In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy import stats

In [2]:
df = pd.read_csv("diabetes.csv")
df.head()
df.shape

(100, 9)

In [3]:
df.loc[5, 'Glucose'] = np.nan
df.loc[10, 'BMI'] = np.nan
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,5,118.0,79,10,106,23.9,1.314,38,1
1,0,163.0,49,18,126,30.8,0.845,26,1
2,3,168.0,113,25,241,47.7,0.995,57,0
3,3,112.0,81,25,240,21.2,1.360,22,1
4,7,182.0,63,21,146,48.2,1.901,50,0


In [4]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               100 non-null    int64  
 1   Glucose                   99 non-null     float64
 2   BloodPressure             100 non-null    int64  
 3   SkinThickness             100 non-null    int64  
 4   Insulin                   100 non-null    int64  
 5   BMI                       99 non-null     float64
 6   DiabetesPedigreeFunction  100 non-null    float64
 7   Age                       100 non-null    int64  
 8   Outcome                   100 non-null    int64  
dtypes: float64(3), int64(6)
memory usage: 7.2 KB
None


In [5]:
print(df.describe())

       Pregnancies     Glucose  BloodPressure  SkinThickness     Insulin  \
count   100.000000   99.000000     100.000000     100.000000  100.000000   
mean      4.330000  133.555556      78.360000      35.320000  156.790000   
std       2.821473   40.656406      22.324747      14.662451   70.908149   
min       0.000000   70.000000      40.000000      10.000000   15.000000   
25%       2.000000   99.500000      60.750000      24.500000  108.000000   
50%       4.000000  131.000000      76.000000      35.000000  151.500000   
75%       7.000000  174.000000      98.750000      48.250000  221.250000   
max       9.000000  199.000000     115.000000      59.000000  271.000000   

             BMI  DiabetesPedigreeFunction         Age     Outcome  
count  99.000000                100.000000  100.000000  100.000000  
mean   34.044444                  1.303580   48.700000    0.590000  
std     9.672866                  0.717235   15.984525    0.494311  
min    18.400000                  0.106

In [6]:
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

Glucose    1
BMI        1
dtype: int64


In [7]:
imputer = SimpleImputer(strategy="median")

cols = ["Glucose","BloodPressure","SkinThickness","Insulin","BMI"]

for col in cols:
    df[col] = df[col].replace(0, np.nan)

df[cols] = imputer.fit_transform(df[cols])

print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [8]:
df_encoded = df.copy()

In [10]:
# Apply Min-Max Scaling to bring values between 0 and 1

normalizer = MinMaxScaler()
df_encoded[['Glucose']] = normalizer.fit_transform(df_encoded[['Glucose']])
df_encoded.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,5,0.372093,79.0,10.0,106.0,23.9,1.314,38,1
1,0,0.720930,49.0,18.0,126.0,30.8,0.845,26,1
2,3,0.759690,113.0,25.0,241.0,47.7,0.995,57,0
3,3,0.325581,81.0,25.0,240.0,21.2,1.360,22,1
4,7,0.868217,63.0,21.0,146.0,48.2,1.901,50,0


In [11]:
# Apply Standardization (mean=0, std=1)
scaler = StandardScaler()
df_encoded[['Age']] = scaler.fit_transform(df_encoded[['Age']])
df_encoded.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,5,0.372093,79.0,10.0,106.0,23.9,1.314,-0.672770,1
1,0,0.720930,49.0,18.0,126.0,30.8,0.845,-1.427278,1
2,3,0.759690,113.0,25.0,241.0,47.7,0.995,0.521868,0
3,3,0.325581,81.0,25.0,240.0,21.2,1.360,-1.678781,1
4,7,0.868217,63.0,21.0,146.0,48.2,1.901,0.081738,0


In [12]:
# Detect and remove outliers using IQR method
df_copy1 = df_encoded.copy()

Q1 = df_copy1.quantile(0.25)
Q3 = df_copy1.quantile(0.75)
IQR = Q3 - Q1

df_copy1 = df_copy1[~((df_copy1 < (Q1 - 1.5 * IQR)) |
                      (df_copy1 > (Q3 + 1.5 * IQR))).any(axis=1)]

df_copy1.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,5,0.372093,79.0,10.0,106.0,23.9,1.314,-0.672770,1
1,0,0.720930,49.0,18.0,126.0,30.8,0.845,-1.427278,1
2,3,0.759690,113.0,25.0,241.0,47.7,0.995,0.521868,0
3,3,0.325581,81.0,25.0,240.0,21.2,1.360,-1.678781,1
4,7,0.868217,63.0,21.0,146.0,48.2,1.901,0.081738,0


In [13]:
# Detect outliers using Z-score and replace with NaN
df_copy2 = df_encoded.copy()

df_copy2['Glucose_zscore'] = stats.zscore(df_copy2['Glucose'])

df_copy2['Glucose'] = np.where(
    df_copy2['Glucose_zscore'].abs() > 3,
    np.nan,
    df_copy2['Glucose']
)

df_copy2.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,Glucose_zscore
0,5,0.372093,79.0,10.0,106.0,23.9,1.314,-0.672770,1,-0.385852
1,0,0.720930,49.0,18.0,126.0,30.8,0.845,-1.427278,1,0.732200
2,3,0.759690,113.0,25.0,241.0,47.7,0.995,0.521868,0,0.856427
3,3,0.325581,81.0,25.0,240.0,21.2,1.360,-1.678781,1,-0.534926
4,7,0.868217,63.0,21.0,146.0,48.2,1.901,0.081738,0,1.204266


In [14]:
# Replace outliers using median value
df_copy3 = df_encoded.copy()

df_copy3['Glucose_zscore'] = stats.zscore(df_copy3['Glucose'])
median_val = df_copy3['Glucose'].median()

df_copy3['Glucose'] = np.where(
    df_copy3['Glucose_zscore'].abs() > 3,
    median_val,
    df_copy3['Glucose']
)

df_copy3.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,Glucose_zscore
0,5,0.372093,79.0,10.0,106.0,23.9,1.314,-0.672770,1,-0.385852
1,0,0.720930,49.0,18.0,126.0,30.8,0.845,-1.427278,1,0.732200
2,3,0.759690,113.0,25.0,241.0,47.7,0.995,0.521868,0,0.856427
3,3,0.325581,81.0,25.0,240.0,21.2,1.360,-1.678781,1,-0.534926
4,7,0.868217,63.0,21.0,146.0,48.2,1.901,0.081738,0,1.204266


In [ ]:
#NOW GOING TO ADULT INCOME DATASET

In [15]:
# Load adult income dataset
df = pd.read_csv("adult.csv")
df.head()

,age,workclass,education,marital-status,occupation,relationship,race,sex,hours-per-week,income
0,25,Private,HS-grad,Single,Tech,Not-in-family,Asian,Male,25,>50K
1,47,Self-emp,HS-grad,Single,Sales,Husband,Asian,Female,18,>50K
2,32,Self-emp,HS-grad,Married,Sales,Not-in-family,Black,Female,22,<=50K
3,29,Gov,Masters,Married,Sales,Not-in-family,White,Female,31,>50K
4,27,Self-emp,Bachelors,Married,Sales,Not-in-family,Black,Male,19,<=50K


In [16]:
# Display dataset information and summary
print(df.info())
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             100 non-null    int64 
 1   workclass       100 non-null    object
 2   education       100 non-null    object
 3   marital-status  100 non-null    object
 4   occupation      100 non-null    object
 5   relationship    100 non-null    object
 6   race            100 non-null    object
 7   sex             100 non-null    object
 8   hours-per-week  100 non-null    int64 
 9   income          100 non-null    object
dtypes: int64(2), object(8)
memory usage: 7.9+ KB
None
              age  hours-per-week
count  100.000000      100.000000
mean    39.590000       31.910000
std     13.778255       17.384014
min     18.000000        2.000000
25%     29.750000       14.500000
50%     36.500000       32.500000
75%     49.000000       48.000000
max     69.000000       59.000000


In [17]:
# Replace '?' with NaN and handle missing values
df.replace("?", np.nan, inplace=True)
print(df.isnull().sum())

# Fill missing values using forward fill
df.fillna(method='ffill', inplace=True)

age               0
workclass         0
education         0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
hours-per-week    0
income            0
dtype: int64


/tmp/ipykernel_1543/3355480069.py:6: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)


In [18]:
# Convert categorical columns into numerical using Label Encoding
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

In [19]:
# Apply Min-Max Scaling on age column
normalizer = MinMaxScaler()
df[['age']] = normalizer.fit_transform(df[['age']])

In [20]:
# Apply Standardization on hours-per-week column
scaler = StandardScaler()
df[['hours-per-week']] = scaler.fit_transform(df[['hours-per-week']])

In [21]:
# Remove outliers using IQR method
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1

df = df[~((df < (Q1 - 1.5 * IQR)) |
          (df > (Q3 + 1.5 * IQR))).any(axis=1)]

df.head()

,age,workclass,education,marital-status,occupation,relationship,race,sex,hours-per-week,income
0,0.137255,1,1,1,2,1,0,1,-0.399494,1
1,0.568627,2,1,1,1,0,0,0,-0.804191,1
2,0.274510,2,1,0,1,1,1,0,-0.572936,0
3,0.215686,0,2,0,1,1,2,0,-0.052611,1
4,0.176471,2,0,0,1,1,1,1,-0.746378,0
